"""Experiment III: nonlinear 2D Neural SDE; CPU-friendly PyTorch.

Run: python experiment3_nonlinear_2d_nsde.py


Dependencies: torch, numpy, matplotlib

Also works in VS Code Jupyter via %run experiment3_nonlinear_2d_nsde.py
"""

Use the enviroment : Pytorch_2026

In [6]:

#%pip install matplotlib

In [7]:
from pathlib import Path
import csv
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 20261022
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 4))
DEVICE = torch.device('cpu')
OUT = Path.cwd() / 'experiment3_results'
OUT.mkdir(parents=True, exist_ok=True)
A = torch.tensor([[-2., -3.], [3., -2.]], device=DEVICE)
W = torch.tensor([[.5, -.2], [.1, .4]], device=DEVICE)
SIGMA = .25
T = 1.0
N_COARSE = 20
REF_SUBSTEPS = 10
N_TRAIN, N_TEST = 192, 64
EPOCHS = 100
BATCH_SIZE = 48
LR = 2e-3


def true_drift(z):
    return z @ A.T + torch.tanh(z @ W.T)


def make_data(npaths, seed):
    """Fine Euler-Maruyama reference; aggregate fine increments to coarse increments."""
    gen = torch.Generator(device='cpu').manual_seed(seed)
    z0 = torch.randn(npaths, 2, generator=gen) * .7
    dt = T / (N_COARSE * REF_SUBSTEPS)
    fine_dw = torch.randn(npaths, N_COARSE, REF_SUBSTEPS, 2, generator=gen) * dt**.5
    z = z0.clone()
    trajectory = [z.clone()]
    with torch.no_grad():
        for k in range(N_COARSE):
            for j in range(REF_SUBSTEPS):
                z = z + dt * true_drift(z) + SIGMA * fine_dw[:, k, j, :]
            trajectory.append(z.clone())
    return z0, fine_dw.sum(dim=2), torch.stack(trajectory, dim=1)


class MLP(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, width), nn.Tanh(), nn.Linear(width, 2))
    def forward(self, z):
        return self.net(z)


class NeuralSDE(nn.Module):
    def __init__(self, scheme):
        super().__init__()
        self.scheme = scheme
        self.net = MLP()
        if scheme == 'Monolithic':
            # The whole drift is learned without being given A.
            self.drift = self.net
        else:
            # Known A plus a learned nonlinear residual.
            self.drift = lambda z: z @ A.T + self.net(z)
        self.register_buffer('linear_flow', torch.matrix_exp(A * (T / N_COARSE)))
        self.register_buffer('linear_half', torch.matrix_exp(A * (T / (2 * N_COARSE))))

    def nonlinear_rk2(self, z, dt):
        k1 = self.net(z)
        return z + .5 * dt * (k1 + self.net(z + dt * k1))

    def step(self, z, dw):
        h = T / N_COARSE
        if self.scheme == 'Monolithic':
            # Euler-Maruyama with learned full drift.
            return z + h * self.drift(z) + SIGMA * dw
        if self.scheme == 'Lie':
            # Exact linear flow -> nonlinear RK2 -> additive-noise flow.
            z = z @ self.linear_flow.T
            z = self.nonlinear_rk2(z, h)
            return z + SIGMA * dw
        if self.scheme == 'Strang':
            # Symmetric deterministic composition around the additive-noise flow:
            # A/2 -> B/2 -> noise -> B/2 -> A/2.
            # This is a numerical architecture, not a guarantee of strong order 2.
            z = z @ self.linear_half.T
            z = self.nonlinear_rk2(z, h / 2)
            z = z + SIGMA * dw
            z = self.nonlinear_rk2(z, h / 2)
            return z @ self.linear_half.T
        raise ValueError(self.scheme)

    def forward(self, z0, dw):
        z = z0
        states = [z]
        for k in range(dw.shape[1]):
            z = self.step(z, dw[:, k, :])
            states.append(z)
        return torch.stack(states, dim=1)


def train_model(name, data):
    z0, dw, target = data
    model = NeuralSDE(name).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    gen = torch.Generator().manual_seed(SEED + 9)
    history = []
    start = time.perf_counter()
    for epoch in range(EPOCHS):
        order = torch.randperm(len(z0), generator=gen)
        total = 0.
        for idx in order.split(BATCH_SIZE):
            optimizer.zero_grad()
            pred = model(z0[idx], dw[idx])
            loss = (pred - target[idx]).square().mean()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        history.append(total / len(z0))
        if (epoch + 1) % 25 == 0:
            print(f'{name:11s} epoch {epoch+1:3d}/{EPOCHS}: MSE={history[-1]:.6f}', flush=True)
    return model, history, time.perf_counter() - start


def metrics(model, data):
    z0, dw, reference = data
    with torch.no_grad():
        predicted = model(z0, dw)
        error = predicted - reference
        terminal = predicted[:, -1, :]
        exact = reference[:, -1, :]
        return {
            'path_rmse': torch.sqrt(error.square().mean()).item(),
            'terminal_rmse': torch.sqrt((terminal - exact).square().mean()).item(),
            'terminal_mean_error': torch.linalg.vector_norm(terminal.mean(0)-exact.mean(0)).item(),
            'terminal_variance_error': torch.linalg.vector_norm(terminal.var(0, unbiased=False)-exact.var(0, unbiased=False)).item(),
        }, predicted


def main():
    print('Creating common-noise reference data...')
    train = make_data(N_TRAIN, SEED + 1)
    test = make_data(N_TEST, SEED + 2)
    rows, histories, predictions = [], {}, {}
    for name in ('Monolithic', 'Lie', 'Strang'):
        model, history, seconds = train_model(name, train)
        values, prediction = metrics(model, test)
        histories[name] = history
        predictions[name] = prediction.numpy()
        row = {'method': name, 'parameters': sum(p.numel() for p in model.parameters()),
               'training_seconds': seconds, **values}
        rows.append(row)
        print('TEST:', row, flush=True)
        torch.save(model.state_dict(), OUT / f'{name.lower()}_weights.pt')
    with (OUT / 'metrics.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    plt.figure(figsize=(7, 4))
    for name, history in histories.items():
        plt.semilogy(np.arange(1, len(history)+1), history, label=name)
    plt.xlabel('Epoch'); plt.ylabel('Training trajectory MSE')
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(OUT / 'training_losses.png', dpi=180); plt.close()
    reference = test[2].numpy()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(reference[0,:,0], reference[0,:,1], 'k-', lw=2, label='Reference')
    for name, pred in predictions.items():
        ax.plot(pred[0,:,0], pred[0,:,1], '--', label=name)
    ax.set(xlabel='Z1', ylabel='Z2', title='Test trajectory: shared Brownian path')
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(OUT / 'test_trajectory.png', dpi=180); plt.close(fig)
    print('\nSaved results to:', OUT)
    return rows

if __name__ == '__main__':
    main()


Creating common-noise reference data...
Monolithic  epoch  25/100: MSE=0.081628
Monolithic  epoch  50/100: MSE=0.010756
Monolithic  epoch  75/100: MSE=0.002856
Monolithic  epoch 100/100: MSE=0.002675
TEST: {'method': 'Monolithic', 'parameters': 162, 'training_seconds': 1.0173454999894602, 'path_rmse': 0.04561087489128113, 'terminal_rmse': 0.04274811968207359, 'terminal_mean_error': 0.003320240881294012, 'terminal_variance_error': 0.006529147271066904}
Lie         epoch  25/100: MSE=0.000123
Lie         epoch  50/100: MSE=0.000123
Lie         epoch  75/100: MSE=0.000122
Lie         epoch 100/100: MSE=0.000122
TEST: {'method': 'Lie', 'parameters': 162, 'training_seconds': 2.046825199999148, 'path_rmse': 0.01207114476710558, 'terminal_rmse': 0.012815607711672783, 'terminal_mean_error': 0.0035482507664710283, 'terminal_variance_error': 0.0023554451763629913}
Strang      epoch  25/100: MSE=0.000046
Strang      epoch  50/100: MSE=0.000045
Strang      epoch  75/100: MSE=0.000044
Strang      e